# LTX 2.5 Fast

<p align="center">
  <img src="../../.github/images/banners/fal-ltx-2.5-fast.png" alt="LTX 2.5 Fast: a social clip for every category, drafted fast" width="900" />
</p>

Fast text-to-video and image-to-video with clips up to 20 seconds, up to 4K, and a range of frame rates, from Lightricks through fal on Microsoft Foundry.

| | |
|---|---|
| Released | 2026-11-06 (preview) |
| Publisher | fal (model by Lightricks) |
| Capability | [Video Generation](../../3-primers/video-generation.md) |
| Model card | [Fal-LTX-2.5-Fast](https://ai.azure.com/catalog/models/Fal-LTX-2.5-Fast) |
| Announcement | [Release post](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/building-the-next-generation-of-multimodal-ai-experiences-with-fal-in-microsoft-/4570032) |
| Docs | [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) |
| Deployment | Global Standard, pay-as-you-go |
| Modes | Text-to-video, image-to-video (start and optional end frame) |
| Clip shape | 6–20 s (even steps) or `auto` · 720p, 1080p, 1440p, 2160p · 24, 25, 48, 50 fps · 16:9, 9:16 · audio on by default |
| Limits that interact | At 720p or 1080p, 24/25 fps allows up to 20 s and 48/50 fps up to 10 s. At 1440p or 2160p, every frame rate allows up to 10 s |
| Rate limit | 2 requests per minute on quota tiers 1–4 |
| Pricing | Not on the model card yet. Check your deployment in the Foundry portal; fal's direct price is on [fal.ai](https://fal.ai/models/fal-ai/ltx-2.5/text-to-video/fast) |
| Env vars | `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_LTX_25_FAST_DEPLOYMENT`, `DEMO_MODE` (optional: `AZURE_FAL_LTX_25_PRO_DEPLOYMENT`) |

**Read first:** the [Video Generation primer](../../3-primers/video-generation.md) explains the terms we use here (duration, resolution, frame rate, aspect ratio, and the cost, latency, and quality trade-offs). Ten minutes there makes this notebook much easier.

**First time here?** Run [setup.ipynb](../../setup.ipynb) once to create your Microsoft Foundry project and `.env`.

> *"Can LTX 2.5 Fast give us a social clip for every product category before lunch, and still make the hero asset?"*

Contoso Outdoors sells gear in seven categories. The social team wants a vertical clip for each one this week, a long take for the website, and a 4K loop for the in-store screen. LTX 2.5 Fast is the speed tier of the LTX family. Its sibling, [LTX 2.5 Pro](../fal-ltx-2.5-pro/), is the quality tier. We use the [Contoso Outdoors catalog](../../2-scenarios/contoso-retail/) and launch images made by an earlier MAI workshop.

## What We Will Do

- Time a single draft, then batch a clip for every category within the rate limit
- Animate a vertical launch image, render a 20-second take, and a 4K loop
- Learn which resolution, frame-rate, and duration combinations are allowed
- Draft on Fast and finish on Pro, then package what we learn for a customer conversation

> Run each cell with **Shift + Enter**. Guess first: before each video, predict one thing it will get right and one it will get wrong.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Before you start

**Just reading?** Leave `DEMO_MODE=replay`. The notebook shows clips saved in `examples/` and makes no Azure calls.

**Running it live?** We need four things:

1. **fal enabled on the subscription.** An admin registers the **Fal.EnableDeploy** preview feature ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#enable-fal-on-foundry)). It can take 30 minutes.
2. **An LTX 2.5 Fast deployment** (Global Standard) in our Foundry project ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#deploy-fal-models-from-the-foundry-portal)). Section 6 also uses **LTX 2.5 Pro** if it's deployed.
3. **A sign-in with a Foundry role** on the resource: run `az login` in a terminal. We use keyless Microsoft Entra ID tokens, never API keys.
4. **`.env` at the repo root** with `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_LTX_25_FAST_DEPLOYMENT`, and `DEMO_MODE=live`. See [sample.env](../../1-scripts/sample.env).

**Data handling.** fal hosts this model outside Microsoft. Prompts and images we send leave Microsoft, and Microsoft's product terms and DPA don't apply. We only send fictional Contoso data here.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "1-scripts" / "fal_video.py").exists())
load_dotenv(ROOT / ".env")

DEMO_MODE = None  # Set to "live" or "replay" to override .env
MODE = (DEMO_MODE or os.getenv("DEMO_MODE") or "replay").strip().lower()
ENDPOINT = os.getenv("MICROSOFT_FOUNDRY_ENDPOINT", "").strip()
DEPLOYMENT = os.getenv("AZURE_FAL_LTX_25_FAST_DEPLOYMENT", "Fal-LTX-2.5-Fast").strip()
PRO_DEPLOYMENT = os.getenv("AZURE_FAL_LTX_25_PRO_DEPLOYMENT", "").strip()
RPM = int(os.getenv("FAL_REQUESTS_PER_MINUTE", "2"))
PLAY_INLINE = os.getenv("PLAY_INLINE", "true").strip().lower() == "true"

if MODE not in ("replay", "live"):
    raise SystemExit(f'DEMO_MODE is "{MODE}". Use "replay" or "live".')

credential = None
if MODE == "live":
    if not ENDPOINT.startswith("https://"):
        raise SystemExit("MICROSOFT_FOUNDRY_ENDPOINT is missing from .env. Run setup.ipynb, or copy it from the project's Overview page.")
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    try:
        credential.get_token("https://cognitiveservices.azure.com/.default")
    except Exception as error:
        raise SystemExit(f"Couldn't get a Microsoft Entra token. Run `az login` in a terminal, then run this cell again.\n{error}")

print(f"Mode: {MODE} | Deployment: {DEPLOYMENT} | Pro for section 6: {PRO_DEPLOYMENT or 'not set'} | {RPM} requests per minute")

The shared helper in [1-scripts/fal_video.py](../../1-scripts/fal_video.py) submits each job, polls until it's done, downloads the MP4, and saves a JSON record. It also **spaces requests** to match the rate limit, which matters in section 2. LTX routes end in the tier: `text-to-video/fast`, `image-to-video/fast`.

In [ ]:
import json

sys.path.insert(0, str(ROOT / "1-scripts"))
from fal_video import FalVideo, as_url, scoreboard

HERE = ROOT / "4-models" / "fal-ltx-2.5-fast"
DATA = ROOT / "2-scenarios" / "contoso-retail" / "data"
IMAGES = DATA / "generated" / "images"

fal = FalVideo(ENDPOINT, DEPLOYMENT, family="lightricks/ltx-2.5", here=HERE, mode=MODE,
               requests_per_minute=RPM, play_inline=PLAY_INLINE, credential=credential)
clips = []

products = json.loads((DATA / "products.json").read_text(encoding="utf-8"))
first_in_category = {}
for p in products:
    first_in_category.setdefault(p["category"], p)
for category, p in first_in_category.items():
    print(f"{category:16} {p['name']}")

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 1. How fast is one draft?

A draft is for judging an idea, not for shipping. We use the cheapest useful settings: 720p, 6 seconds, 25 fps, vertical. The "made in" figure on the facts line is the number to remember.

> 🔮 **Guess first:** how many seconds from submit to download? Under 30? Over 60?

In [ ]:
def social_prompt(p: dict) -> str:
    return (f"Vertical social clip: the {p['name']} by {p['brand']} in use on a sunny trail weekend. "
            "Quick, energetic handheld moves, bright natural light, upbeat outdoor-brand feel. "
            "Audio: nature sounds and a light upbeat rhythm.")

DRAFT = dict(duration=6, resolution="720p", fps=25, aspect_ratio="9:16")
draft = fal.run("01-draft-tents", "text-to-video/fast", prompt=social_prompt(first_in_category["Tents"]), **DRAFT)
clips.append(draft)
fal.show(draft)

**What to look for:** the "made in" time against the clip's own length. If a 6-second clip takes about as long to make as it takes to watch, iteration feels fast. Then judge the idea, not the polish: is "in use on a trail" a good brief for a tent at all?

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 2. Can we make a clip for every category?

Seven categories, seven clips, one template. At 2 requests per minute the helper waits about 30 seconds between submits, so the batch takes a few minutes: that's the rate limit, not a bug. (We already have the tent clip from section 1.)

In [ ]:
batch = [draft]
for category, p in first_in_category.items():
    if category == "Tents":
        continue
    slug = category.lower().replace(" ", "-")
    batch.append(fal.run(f"02-social-{slug}", "text-to-video/fast", prompt=social_prompt(p), **DRAFT))
clips += batch[1:]
fal.show(*batch, frames=4)

**What to look for:** does each clip show the *right kind* of product (a stove for Camping Stoves, boots for Hiking Footwear)? Which categories fit the template, and which need their own prompt? For volume work, the template is the product: fix it once, rerun the batch.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 3. Can we animate a vertical launch image?

The earlier MAI workshop made a 9:16 "phone story" image: the tent under the night sky. With image-to-video and `aspect_ratio="auto"`, the clip follows the image's shape.

In [ ]:
from IPython.display import Image as Picture, display

display(Picture(filename=str(IMAGES / "phone-story.webp"), width=200))

story = fal.run("03-phone-story", "image-to-video/fast", image_url=as_url(IMAGES / "phone-story.webp"),
                prompt="Stars wheel slowly overhead; a warm light flickers on inside the tent. Static camera. Audio: crickets, soft wind.",
                aspect_ratio="auto", duration=6, resolution="720p", fps=25)
clips.append(story)
fal.show(story)

**What to look for:** the facts line should report a vertical size (taller than wide). Check the first frame against the image and whether the stars move *together*, the way a real sky turns.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 4. How long can one take be?

Fast goes to **20 seconds**, but only at 720p or 1080p and 24 or 25 fps. We use the full length for a day-to-night take for the website.

In [ ]:
DAY_TO_NIGHT = ("One continuous take: the TrailMaster X4 tent beside a lake, from bright afternoon through sunset to a starry night. "
                "Shadows lengthen, the sky turns orange then deep blue, a lantern glows inside the tent. Very slow dolly in. "
                "Audio: birds fading into crickets, gentle water.")

long_take = fal.run("04-twenty-seconds", "text-to-video/fast", prompt=DAY_TO_NIGHT, duration=20, resolution="720p", fps=24, aspect_ratio="16:9")
clips.append(long_take)
fal.show(long_take, frames=10)

**What to look for:** the tent in frame 1 vs frame 10. Long takes give the model more time to drift: shape, color, and position. If the tent changes, split the take into shorter clips with shared first and last frames.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 5. Can it make a 4K loop for the store?

The in-store screen is 4K. Fast renders **2160p**, up to 10 seconds. We start from the campsite image so the scene is ours, and keep the sound off because the store plays its own music.

In [ ]:
uhd = fal.run("05-4k-store-loop", "image-to-video/fast", image_url=as_url(IMAGES / "campsite-before.webp"),
              prompt="Gentle ripples cross the lake; leaves sway; the tent fabric breathes in the breeze. Static camera.",
              duration=6, resolution="2160p", fps=25, generate_audio=False)
clips.append(uhd)
fal.show(uhd)

**What to look for:** the size (3840×2160 for 16:9) and the file size on the facts line, and the "made in" time compared with the 720p clips. 4K is for the final screen, not for drafts. Our start image is only 1536×1024, so the model is inventing detail; zoom in on the tent's seams to see how convincing that is.

### 🔧 Break it on purpose

The limits interact: at 2160p the maximum is 10 seconds. What happens if we ask for 20?

In [ ]:
if MODE == "live":
    try:
        fal.run("break-2160p-20s", "text-to-video/fast", prompt=DAY_TO_NIGHT, duration=20, resolution="2160p")
    except RuntimeError as error:
        print(error)
else:
    print("Replay mode: in live mode the service rejects the request and explains that 1440p and 2160p allow up to 10 seconds.")

Each setting is valid on its own; the **combination** isn't. Read the table in At a glance as a grid, not a list. Validating combinations before submitting saves a round trip for every bad request in a batch.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 6. Draft on Fast, finish on Pro?

The primer's pattern: explore cheap, render the winner on the quality tier. We take the tent clip from section 1 and render the same prompt on **LTX 2.5 Pro** at 1080p. This section runs only if `AZURE_FAL_LTX_25_PRO_DEPLOYMENT` is set.

In [ ]:
if PRO_DEPLOYMENT or MODE == "replay":
    pro = FalVideo(ENDPOINT, PRO_DEPLOYMENT or "Fal-LTX-2.5-Pro", family="lightricks/ltx-2.5", here=HERE, mode=MODE,
                   requests_per_minute=RPM, play_inline=PLAY_INLINE, credential=credential)
    final = pro.run("06-final-on-pro", "text-to-video/pro", prompt=social_prompt(first_in_category["Tents"]),
                    duration=6, resolution="1080p", fps=25, aspect_ratio="9:16")
    clips.append(final)
    fal.show(draft, final)
else:
    print("Set AZURE_FAL_LTX_25_PRO_DEPLOYMENT in .env to run this comparison.")

**What to look for:** the two clips aren't the same video at higher quality. Each is a new generation, so composition and action can change. To carry the *idea* across, keep the prompt; to carry the *look*, take a frame you like from the draft and use it as the Pro clip's start image.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 7. What did each clip cost us in time?

### Scoreboard

Every clip we made, from its saved record: how long the job took end to end, the duration we asked for vs the one we got, and the output's shape and size.

In [ ]:
scoreboard(clips)

**What to look for:** the spread between the 720p drafts, the 20-second take, and the 4K loop, both in render time and in megabytes. How long did the seven-clip batch take in total, and how much of that was waiting for the rate limit? For pricing, use the deployment's page in the Foundry portal, not estimates.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Teach it to a customer

### What it does, and why it matters

- **The fast tier of LTX 2.5**: quick drafts and volume work, with synchronized audio.
- **The widest range in the family:** up to 20 seconds, up to 2160p (4K), 24–50 fps, and `auto` duration.
- **Image-to-video with optional end frame**, and the same camera presets as Pro.
- **A natural pair with Pro**: draft and batch on Fast, render hero assets on Pro.

### A prompt recipe that works

Build a **template** from catalog fields (`name`, `brand`, `category`) plus a fixed style line, then batch it. Change the template, not individual prompts, so every clip stays on brand.

### Honest answers to common questions

- **Is it production-ready?** It's in public preview on Foundry. Validate quality and terms before production.
- **Where does our data go?** To fal, outside Microsoft. Microsoft's product terms, DPA, and certifications don't apply. Review [fal's terms](https://fal.ai/legal) with the customer.
- **How many clips an hour?** At 2 requests per minute (tiers 1–4), at most 120 submits an hour; tiers 5–7 allow 40 per minute.
- **Is 4K "real" 4K?** It's rendered at 2160p, but detail comes from the model; check it against the source.

### When something goes wrong

- **422 for a resolution, fps, and duration combination** → check the grid: 20 s needs 720p or 1080p at 24 or 25 fps.
- **429 during a batch** → the helper retries; lower `FAL_REQUESTS_PER_MINUTE` if it keeps happening.
- **A category clip shows the wrong product** → add the category's defining object to the template for that category.
- **Long takes drift** → shorten the clip, or chain shorter clips through shared start and end frames.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Your Turn to Explore

- **Faster than fast.** Rerun section 1 with `duration="auto"`. Does the model pick something shorter than 6 seconds?
- **48 vs 50 fps.** Fast also offers 48 fps (Pro doesn't). Render the section 4 prompt at 48 fps and 10 seconds.
- **A seasonal set.** Animate [four-seasons.webp](../../2-scenarios/contoso-retail/data/generated/images/four-seasons.webp) with a prompt that moves through each panel.

In [ ]:
# Your experiment here. Try: fal.run("my-clip", "text-to-video/fast", prompt="...", duration=6, resolution="720p")

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## What We Did

- ✅ Timed a single draft, then batched a clip for every category within the rate limit
- ✅ Animated a vertical launch image, rendered a 20-second take, and a 4K loop
- ✅ Learned which resolution, frame-rate, and duration combinations are allowed
- ✅ Drafted on Fast, finished on Pro, and packaged the findings for a customer conversation

➡️ **Next:** [MiniMax H3 Max Turbo](../fal-minimax-h3-max-turbo/) is another fast tier, with first and last frames and a pinned soundtrack.

Stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

**Learn more:** [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) · [Video Generation primer](../../3-primers/video-generation.md) · [LTX 2.5 Fast API on fal](https://fal.ai/models/fal-ai/ltx-2.5/text-to-video/fast/api)